# 🧠 MemoRIA — Memoria Personal Semántica

> **Proyecto:** Engineering Project — Curso NLP, MIA, UdeSA 2026  
> **Equipo:** Nico Karagozian, Clara Kearney, Valen Pivotto  
> **Profesor:** Luciano Del Corro  
> **Modelo base:** `google/gemma-4-E2B-it` (Gemma 4 E2B — 2.3B params efectivos / 5.1B total, context 128K)

---

## Pregunta de investigación

> ¿Puede un modelo de 2B parámetros aprender múltiples registros lingüísticos de un individuo usando sus propios datos reales y hardware de consumo?

**MemoRIA** es un LLM pequeño ajustado fino sobre textos personales en tres registros:

| Tag | Registro | Fuente |
|-----|----------|---------|
| `[CASUAL]` | Conversacional / WhatsApp | Exports `.txt` |
| `[EMAIL-PROF]` | Profesional / email | `.mbox` de Google Takeout |
| `[ACADÉMICO]` | Académico / ensayos | PDFs y DOCX personales |

**Lo que NO es:** No es prompting a un LLM grande. Es fine-tuning real con **LoRA**.

---

## Índice
1. [Setup e instalación](#setup)
2. [Datos y preprocesamiento](#datos)
3. [Fine-tuning](#finetuning)
4. [Inferencia local](#inferencia)
5. [Exportar a Ollama](#ollama)
6. [Evaluación](#evaluacion)
7. [Backend FastAPI](#backend)

---
## 1. Setup e Instalación <a id='setup'></a>

**Mac M5 (path principal):** MLX-LM con cuantización 4-bit. Requiere `mlx-lm>=0.22`.
**Colab T4 (fallback):** QLoRA con bitsandbytes. Ver sección 3 para elegir el path.

> ⚠️ **Mac M5:** Cerrar browsers y apps pesadas antes de entrenar para liberar RAM unificada.

In [ ]:
# Instalar dependencias (correr una sola vez)
!pip install -q \
    mlx-lm>=0.22.0 \
    transformers>=4.50.0 \
    peft>=0.14.0 \
    trl>=0.12.0 \
    bitsandbytes>=0.43.0 \
    accelerate>=0.27.0 \
    datasets>=2.20.0 \
    pdfplumber>=0.10.0 \
    python-docx>=1.1.0 \
    html2text>=2024.2.26 \
    nltk>=3.8.0 \
    scikit-learn>=1.4.0 \
    scipy>=1.12.0 \
    tqdm>=4.66.0 \
    httpx>=0.27.0 \
    spacy>=3.7.0

!python -m spacy download es_core_news_sm -q

In [ ]:
import os
import sys
import re
import json
import math
import random
import mailbox
from pathlib import Path
from collections import Counter
from email.header import decode_header

# MPS fallback: operaciones sin kernel MPS nativo caen a CPU en vez de crashear
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import torch
import numpy as np

print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
print(f"MPS disponible:  {torch.backends.mps.is_available()}")

if torch.cuda.is_available():
    print(f"GPU CUDA: {torch.cuda.get_device_name(0)} — {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif torch.backends.mps.is_available():
    print("GPU Metal (Apple Silicon) disponible")

# Agregar el root del proyecto al path para importar scripts/
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
# ─── Configuración global ────────────────────────────────────────────────────

# ← Cambiar estos valores antes de correr
AUTHOR_NAME  = "Nicolas Karagozian"                          # Nombre exacto en WhatsApp
AUTHOR_EMAIL = "nkaragozian@tnplatex.com"     # Email del remitente en Gmail

# Modelo base — Gemma 4 E2B
MODEL_ID       = "google/gemma-4-E2B-it"
MLX_MODEL_PATH = Path("./models/gemma4-e2b-4bit")  # Modelo cuantizado local

# Path de entrenamiento:
#   True  → MLX-LM nativo (Mac M5, ~25-40 min) — RECOMENDADO
#   False → PyTorch MPS sin cuantización (experimental en 16 GB con Gemma 4 E2B)
USE_MLX = True

DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")

# Rutas
DATA_DIR      = Path("data")
RAW_DIR       = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
DATASET_DIR   = DATA_DIR / "dataset"
OUTPUT_DIR    = Path("memoria-lora")

for d in [RAW_DIR / "whatsapp", RAW_DIR / "gmail", RAW_DIR / "academic",
          PROCESSED_DIR, DATASET_DIR, OUTPUT_DIR, MLX_MODEL_PATH.parent]:
    d.mkdir(parents=True, exist_ok=True)

print("✓ Configuración lista")
print(f"  Modelo: {MODEL_ID}")
print(f"  Device: {DEVICE} | USE_MLX: {USE_MLX}")
print(f"  data/raw/whatsapp/ ← copiá tus .txt acá")
print(f"  data/raw/gmail/    ← copiá tu .mbox acá")
print(f"  data/raw/academic/ ← copiá tus PDFs/DOCX acá")

---
## 2. Datos y Pipeline de Preprocesamiento <a id='datos'></a>

| Fuente | Registro | Volumen esperado |
|--------|----------|-----------------|
| WhatsApp (.txt) | Casual | 500–2000 |
| Gmail (.mbox) | Email prof. | 200–800 |
| PDFs/DOCX | Académico | 300–1000 chunks |
| **Total** | | **≥ 1500 ejemplos** |

### Cómo obtener los datos
- **WhatsApp:** Chat → ⋮ → *Exportar chat* → *Sin archivos* → `.txt`
- **Gmail:** myaccount.google.com → *Descargar datos* → Solo Gmail → `.mbox`
- **Académico:** TPs, papers, informes en PDF o DOCX

> ⚠️ **Privacidad:** Los parsers aplican anonimización automática de PII (nombres, teléfonos, emails de terceros) con spaCy antes de guardar al dataset.

In [ ]:
# ─── 2.1 Parser de WhatsApp ───────────────────────────────────────────────────
from scripts.parse_whatsapp import parse_whatsapp

casual_examples = []
for txt_file in (RAW_DIR / "whatsapp").glob("*.txt"):
    print(f"Procesando: {txt_file.name}")
    parsed = parse_whatsapp(str(txt_file), author_name=AUTHOR_NAME)
    casual_examples.extend(parsed)
    print(f"  → {len(parsed)} mensajes")

print(f"\nTotal casual: {len(casual_examples)} ejemplos")

casual_out = PROCESSED_DIR / "casual.jsonl"
with open(casual_out, "w", encoding="utf-8") as f:
    for ex in casual_examples:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")
print(f"Guardado en {casual_out}")

In [ ]:
# ─── 2.2 Parser de Gmail (.mbox) ─────────────────────────────────────────────
from scripts.parse_gmail import parse_mbox

email_examples = []
for mbox_file in (RAW_DIR / "gmail").glob("*.mbox"):
    print(f"Procesando: {mbox_file.name} (puede tardar varios minutos)")
    parsed = parse_mbox(str(mbox_file), sender_email=AUTHOR_EMAIL)
    email_examples.extend(parsed)
    print(f"  → {len(parsed)} emails")

print(f"\nTotal email_prof: {len(email_examples)} ejemplos")

email_out = PROCESSED_DIR / "email_prof.jsonl"
with open(email_out, "w", encoding="utf-8") as f:
    for ex in email_examples:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")
print(f"Guardado en {email_out}")

In [ ]:
# ─── 2.3 Parser académico (PDF / DOCX) ───────────────────────────────────────
from scripts.parse_academic import parse_academic_folder

academic_examples = parse_academic_folder(str(RAW_DIR / "academic"))
print(f"\nTotal academic: {len(academic_examples)} chunks")

academic_out = PROCESSED_DIR / "academic.jsonl"
with open(academic_out, "w", encoding="utf-8") as f:
    for ex in academic_examples:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")
print(f"Guardado en {academic_out}")

In [ ]:
# ─── 2.4 Construcción del dataset final (train / val / test 80/10/10) ─────────
from scripts.build_dataset import build_dataset

train_set, val_set, test_set = build_dataset(
    casual_file=str(PROCESSED_DIR / "casual.jsonl"),
    email_file=str(PROCESSED_DIR / "email_prof.jsonl"),
    academic_file=str(PROCESSED_DIR / "academic.jsonl"),
)

In [ ]:
# Inspección — verificar que el chat template de Gemma 4 está bien aplicado
print("=" * 70)
print("MUESTRA DEL DATASET")
print("=" * 70)
for ex in random.sample(train_set, min(3, len(train_set))):
    print(f"\n[{ex['register'].upper()}]")
    print(ex["text"][:500])
    print("─" * 50)

---
## 3. Fine-tuning <a id='finetuning'></a>

**Gemma 4 E2B** tiene ~5.1B params totales. En Mac M5 16 GB se **requiere** cuantización 4-bit.

| Path | Hardware | Tiempo estimado | Cómo |
|------|----------|----------------|---------|
| **MLX-LM 4-bit** | Mac M5 16 GB | ~25–40 min | `USE_MLX = True` |
| PyTorch MPS | Mac M5 16 GB | ~3–5 h (límite de RAM) | `USE_MLX = False` |
| QLoRA bitsandbytes | Colab T4 | ~3–4 h | Sección 3 legacy |

> ⚠️ **Importante:** El path MLX cuantiza el modelo a 4-bit antes de entrenar (paso único, queda en disco). Sin cuantización, Gemma 4 E2B no cabe en 16 GB.

In [ ]:
# ─── 3.1 Paso 0 (solo MLX): cuantizar modelo base a 4-bit ────────────────────
# Correr solo una vez — deja el modelo cuantizado en ./models/gemma4-e2b-4bit

if USE_MLX:
    if not MLX_MODEL_PATH.exists() or not any(MLX_MODEL_PATH.iterdir()):
        print(f"Descargando y cuantizando {MODEL_ID} a 4-bit...")
        print("(~5-10 min dependiendo del ancho de banda, queda en disco)")
        !mlx_lm.convert \
            --hf-path {MODEL_ID} \
            --mlx-path {MLX_MODEL_PATH} \
            --quantize --q-bits 4 --q-group-size 64
        print(f"✓ Modelo cuantizado en {MLX_MODEL_PATH}")
    else:
        print(f"✓ Modelo ya existe en {MLX_MODEL_PATH} — saltando conversión")
else:
    print("USE_MLX=False — se usará PyTorch MPS (ver celda siguiente)")

In [ ]:
# ─── 3.2 Fine-tuning con MLX-LM (USE_MLX=True) ───────────────────────────────
# ⚠️ Esta celda tarda ~25-40 min en M5. Monitorear RAM en Activity Monitor.

if USE_MLX:
    !mlx_lm.lora \
        --model {MLX_MODEL_PATH} \
        --train \
        --data data/dataset \
        --iters 1000 \
        --batch-size 1 \
        --num-layers 16 \
        --learning-rate 2e-4 \
        --adapter-path {OUTPUT_DIR} \
        --grad-checkpoint \
        --save-every 200
    print(f"\n✓ Adaptador LoRA guardado en {OUTPUT_DIR}")
else:
    print("USE_MLX=False — ver celda de PyTorch MPS")

In [ ]:
# ─── 3.3 Fine-tuning con PyTorch MPS (USE_MLX=False, experimental) ───────────
# ⚠️ Gemma 4 E2B en 16 GB unificados está al límite. Cerrar todo antes de correr.

if not USE_MLX:
    from transformers import (
        AutoTokenizer, AutoModelForCausalLM, TrainingArguments
    )
    from peft import LoraConfig, get_peft_model
    from trl import SFTConfig, SFTTrainer
    from trl.trainer import DataCollatorForCompletionOnlyLM
    from datasets import Dataset

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    print(f"Cargando {MODEL_ID} en bfloat16 en MPS...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map={"":"mps"},
    )

    lora_config = LoraConfig(
        r=16, lora_alpha=32,
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    collator = DataCollatorForCompletionOnlyLM(
        response_template="<start_of_turn>model\n",
        tokenizer=tokenizer,
    )

    def load_hf(path):
        data = [json.loads(l) for l in open(path) if l.strip()]
        return Dataset.from_list(data)

    sft_config = SFTConfig(
        output_dir=str(OUTPUT_DIR),
        num_train_epochs=3,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=16,
        gradient_checkpointing=True,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        fp16=False, bf16=True,
        logging_steps=10,
        eval_strategy="steps", eval_steps=50,
        save_strategy="steps", save_steps=100,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        report_to="none",
        dataloader_num_workers=0,
        max_seq_length=1024,
        dataset_text_field="text",
    )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=load_hf(DATASET_DIR / "train.jsonl"),
        eval_dataset=load_hf(DATASET_DIR / "val.jsonl"),
        data_collator=collator,
        args=sft_config,
    )

    print("Iniciando entrenamiento MPS...")
    trainer.train()
    trainer.save_model(str(OUTPUT_DIR))
    tokenizer.save_pretrained(str(OUTPUT_DIR))
    print(f"✓ Adaptador LoRA guardado en {OUTPUT_DIR}")
else:
    print("USE_MLX=True — esta celda no corre")

---
## 4. Inferencia local <a id='inferencia'></a>

In [ ]:
# ─── 4. Inferencia con el adaptador entrenado ─────────────────────────────────

test_cases = [
    ("casual",     "Contame cómo estuvo el finde"),
    ("email_prof", "Escribí un email pidiendo una reunión para el martes"),
    ("academic",   "Explicá el concepto de fine-tuning en NLP"),
]

if USE_MLX:
    print("Inferencia con MLX-LM:")
    for register, prompt in test_cases:
        tag = {"casual":"[CASUAL]","email_prof":"[EMAIL-PROF]","academic":"[ACADÉMICO]"}[register]
        print(f"\n{'='*65}")
        print(f"Registro: {register.upper()} | Prompt: {prompt}")
        print("─"*65)
        !mlx_lm.generate \
            --model {MLX_MODEL_PATH} \
            --adapter-path {OUTPUT_DIR} \
            --prompt "{tag} {prompt}" \
            --max-tokens 300 --temp 0.8 --top-p 0.9
else:
    from scripts.inference import load_model, generate
    print("Cargando modelo fine-tuneado (PyTorch MPS)...")
    ft_model, ft_tokenizer = load_model(MODEL_ID, str(OUTPUT_DIR))

    for register, prompt in test_cases:
        print(f"\n{'='*65}")
        print(f"Registro: {register.upper()} | Prompt: {prompt}")
        print("─"*65)
        print(generate(ft_model, ft_tokenizer, register, prompt))

---
## 5. Exportar a Ollama <a id='ollama'></a>

In [ ]:
# ─── 5.1 Merge del adaptador LoRA con el modelo base ─────────────────────────
MERGED_DIR = Path("memoria-merged")

if USE_MLX:
    print("Mergeando con mlx_lm.fuse...")
    !mlx_lm.fuse \
        --model {MLX_MODEL_PATH} \
        --adapter-path {OUTPUT_DIR} \
        --save-path {MERGED_DIR} \
        --de-quantize
    print(f"✓ Modelo mergeado en {MERGED_DIR} (bf16, listo para GGUF)")
else:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from peft import PeftModel
    print("Mergeando con merge_and_unload (CPU)...")
    base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map="cpu")
    tok  = AutoTokenizer.from_pretrained(MODEL_ID)
    base = PeftModel.from_pretrained(base, str(OUTPUT_DIR))
    base = base.merge_and_unload()
    base.save_pretrained(str(MERGED_DIR))
    tok.save_pretrained(str(MERGED_DIR))
    print(f"✓ Modelo mergeado en {MERGED_DIR}")

In [ ]:
# ─── 5.2 Convertir a GGUF y registrar en Ollama ───────────────────────────────

!git clone https://github.com/ggerganov/llama.cpp --depth=1 2>/dev/null || echo "llama.cpp ya existe"
!pip install -q -r llama.cpp/requirements.txt

!python llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} \
    --outfile memoria-q4.gguf \
    --outtype q4_k_m

print("✓ GGUF generado: memoria-q4.gguf")
print("\nPara registrar en Ollama, correr en la terminal:")
print("  ollama create memoria -f Modelfile")
print("  ollama run memoria \"[EMAIL-PROF] Escribí un email para Luciano Del Corro\"")

---
## 6. Evaluación <a id='evaluacion'></a>

| Nivel | Qué mide | Criterio de éxito |
|-------|----------|-------------------|
| E1 Perplexidad | Probabilidad asignada al texto real | Mejora ≥ 20% vs. base |
| E2 Estilo | TTR, argentinismos, longitud de oraciones | Diff < 20% |
| E3 Clasificador BETO | ¿Se puede distinguir real de generado? | Accuracy < 75% |
| E4 Test ciego humano | Jueces conocidos vs. texto real | % correctas < 65% |

> **Nota metodológica:** E3 y E4 usan prompts del catálogo `data/prompts/` — **no** el inicio del texto real — para medir estilo y no continuación.

In [ ]:
# ─── E1: Perplexidad sobre el TEST SET ───────────────────────────────────────
# Se usa test.jsonl (no val) — separación limpia de evaluación

from eval.perplexity import eval_perplexity

results = eval_perplexity(
    test_file=str(DATASET_DIR / "test.jsonl"),
    model_id=MODEL_ID,
    adapter_path=str(OUTPUT_DIR),
)

In [ ]:
# ─── E2: Métricas de estilo ────────────────────────────────────────────────────
import subprocess, re as _re
from eval.style_metrics import compare_styles
from scripts.inference import load_model as load_hf, generate as gen_hf

# Cargar textos del test set agrupados por registro
test_by_register = {"casual": [], "email_prof": [], "academic": []}
with open(DATASET_DIR / "test.jsonl") as f:
    for line in f:
        item = json.loads(line)
        if item["register"] in test_by_register:
            test_by_register[item["register"]].append(item["original_text"])

print("Textos en test set por registro:")
for reg, texts in test_by_register.items():
    print(f"  {reg}: {len(texts)}")

# Generar con prompts del catálogo independiente
prompts_casual = [l.strip() for l in open("data/prompts/casual.txt") if l.strip()][:10]

if USE_MLX:
    # generate_fn_mlx: usa el modelo mergeado vía subprocess (disponible tras merge-lora)
    def generate_fn_mlx(register: str, prompt: str, max_new_tokens: int = 200) -> str:
        _tag = {"casual": "[CASUAL]", "email_prof": "[EMAIL-PROF]", "academic": "[ACADÉMICO]"}.get(register, "[CASUAL]")
        _full = f"{_tag} {prompt}"
        r = subprocess.run(
            ["python", "-m", "mlx_lm", "generate",
             "--model", str(MERGED_DIR),
             "--prompt", _full,
             "--max-tokens", str(max_new_tokens),
             "--temp", "0.8", "--top-p", "0.9"],
            capture_output=True, text=True,
        )
        out = r.stdout.strip()
        if _full in out:
            out = out.split(_full, 1)[-1].strip()
        return _re.sub(r'<[^>]+>', '', out).strip()

    generated_casual = [generate_fn_mlx("casual", p) for p in prompts_casual]
    real_casual = test_by_register["casual"][:len(generated_casual)]
    if real_casual and generated_casual:
        compare_styles(real_casual, generated_casual, register="casual")
else:
    ft_model, ft_tokenizer = load_hf(MODEL_ID, str(OUTPUT_DIR))
    generated_casual = [gen_hf(ft_model, ft_tokenizer, "casual", p, 100) for p in prompts_casual]
    real_casual = test_by_register["casual"][:len(generated_casual)]
    if real_casual and generated_casual:
        compare_styles(real_casual, generated_casual, register="casual")

In [ ]:
# ─── E3: Clasificador de autoría (BETO) ───────────────────────────────────────
# El clasificador debe FALLAR (accuracy < 75%) → el modelo es indistinguible
# Los textos generados usan prompts del catálogo, NO el inicio del texto real

from eval.train_classifier import train_authorship_classifier

test_texts_all = []
with open(DATASET_DIR / "test.jsonl") as f:
    for line in f:
        test_texts_all.append(json.loads(line)["original_text"])

if USE_MLX:
    # generate_fn_mlx definida en la celda eval-style; correr esa celda primero
    acc_result = train_authorship_classifier(
        real_texts=test_texts_all,
        generate_fn=generate_fn_mlx,
        n_samples=150,
    )
else:
    def gen_fn(register, prompt):
        return gen_hf(ft_model, ft_tokenizer, register, prompt, max_new_tokens=150)

    acc_result = train_authorship_classifier(
        real_texts=test_texts_all,
        generate_fn=gen_fn,
        n_samples=150,
    )

In [ ]:
# ─── E4: Generar pares para test ciego ───────────────────────────────────────
from eval.generate_blind_pairs import generate_blind_test_pairs

_gen = generate_fn_mlx if USE_MLX else gen_fn

pairs = generate_blind_test_pairs(
    test_file=str(DATASET_DIR / "test.jsonl"),
    generate_fn=_gen,
    n_per_register=10,
    output_file="eval/blind_test_pairs.json",
)
print(f"\nArchivo para jueces: eval/blind_test_pairs.json")
print("Ground truth (NO compartir): eval/blind_test_key.json")

In [ ]:
# ─── Resumen de criterios de éxito ────────────────────────────────────────────
import os
from datetime import datetime

print("""
╔══════════════════════════════════════════════════════════╗
║           RESUMEN CRITERIOS DE ÉXITO — MemoRIA           ║
╠══════════════╦══════════════════════╦═══════════════════════╣
║ Evaluación   ║ Umbral mínimo        ║ Umbral ideal           ║
╠══════════════╬══════════════════════╬═══════════════════════╣
║ E1 Perplejid ║ Mejora ≥ 20% vs base ║ Mejora ≥ 35%          ║
║ E2 Estilo    ║ Diff métricas < 20%  ║ Diff < 10%            ║
║ E3 Clasif.   ║ Accuracy BETO < 75%  ║ Accuracy < 60%        ║
║ E4 Humano    ║ Jueces < 65% correct ║ Jueces < 55% correct  ║
╚══════════════╩══════════════════════╩═══════════════════════╝""")

# Recoger resultados de las celdas anteriores (si corrieron)
results_summary = {
    "E1_perplexity": results.get("improvement_pct") if "results" in dir() else None,
    "E3_classifier_accuracy": acc_result.get("accuracy") if "acc_result" in dir() else None,
}
print("\nResultados del equipo:")
for k, v in results_summary.items():
    print(f"  {k}: {'⏳ pendiente' if v is None else v}")

# Guardar a archivo para el informe
os.makedirs("eval/results", exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
results_file = f"eval/results/{ts}.json"
with open(results_file, "w") as _f:
    json.dump(results_summary, _f, indent=2, ensure_ascii=False)
print(f"\nResultados guardados en {results_file}")

---
## 7. Backend FastAPI <a id='backend'></a>

El backend completo está en `backend/main.py`. El frontend HTML/JS está en `backend/static/`.

```bash
# Arrancar (requiere Ollama corriendo con el modelo memoria)
uvicorn backend.main:app --host 127.0.0.1 --port 8000 --reload
# Abrir: http://127.0.0.1:8000

# O con Docker Compose:
docker compose up
```

In [ ]:
# Verificar que el backend responde (debe estar corriendo primero)
import httpx

try:
    r = httpx.get("http://127.0.0.1:8000/health", timeout=3)
    data = r.json()
    print(f"Backend: {data['status']}")
    print(f"Modelos en Ollama: {data.get('models', [])}")
except Exception as e:
    print(f"Backend no disponible: {e}")
    print("Arrancar con: uvicorn backend.main:app --host 127.0.0.1 --port 8000")

---
## Próximos pasos

### Sprint 1 (15–21 abr): Datos
- [ ] Exportar 5-8 chats de WhatsApp representativos
- [ ] Google Takeout de Gmail → filtrar Enviados
- [ ] Reunir PDFs/DOCX propios
- [ ] Verificar calidad de 50 ejemplos manualmente por registro
- [ ] **Checkpoint:** ≥ 1500 totales (≥ 400 por registro)

### Sprint 2 (22–28 abr): Fine-tuning
- [ ] Cuantizar modelo con `mlx_lm.convert` (paso único)
- [ ] Correr `bash scripts/finetune_mlx.sh 1000` (~30 min en M5)
- [ ] Verificar inferencia con los 3 tags
- [ ] Merge → GGUF → Ollama
- [ ] **Checkpoint:** el modelo responde de forma diferente por registro

### Sprint 3 (29 abr–5 may): Evaluación
- [ ] E1: Perplexidad sobre test set
- [ ] E2: Métricas de estilo
- [ ] E3: Clasificador BETO
- [ ] E4: Test ciego (3+ jueces que conocen al autor)
- [ ] **Checkpoint:** criterios documentados

### Sprint 4 (6–12 may): MVP + demo
- [ ] Backend FastAPI + frontend corriendo
- [ ] Docker Compose funcional
- [ ] Video demo de 3 min

---

## Stack técnico

| Componente | Herramienta |
|-----------|-------------|
| Modelo base | `google/gemma-4-E2B-it` (Gemma 4 E2B) |
| Fine-tuning (Mac M5) | MLX-LM con cuantización 4-bit |
| Fine-tuning (Colab) | QLoRA / bitsandbytes |
| Inferencia local | Ollama `gemma4:e2b` + modelo fine-tuneado (GGUF Q4_K_M) |
| Anonimización | spaCy `es_core_news_sm` + regex |
| Backend | FastAPI + SSE streaming |
| Frontend | HTML + Vanilla JS + CSS |
| Clasificador | BETO (`dccuchile/bert-base-spanish-wwm-uncased`) |